# 13 Beta Diversity and symptom-related gluten avoidance

We test whether fungal community composition differs between participants who report avoiding gluten because it makes them feel bad and those who report no gluten-related issue (`gluten_sample`). We compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. The adjusted model tests gluten consumption patterns after accounting for age and sex.

## 1. Setup

### 1.1 Imports

In [17]:
import subprocess
from pathlib import Path

import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization, Artifact, Metadata

### 1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [18]:
# 1. Shared inputs and output directories
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/13_beta_diversity_gluten"
vis_dir = "../data/processed/13_beta_diversity_gluten"

columns = ["age_years_sample", "sex_sample", "gluten_sample"]


# 2. Complete-case inputs for both models
adjusted_gluten_out_dir = f"{out_dir}/adjusted_gluten"

adjusted_asv_gluten_distance_path = f"{adjusted_gluten_out_dir}/asv_bray_curtis_complete.qza"
adjusted_kmer_gluten_distance_path = f"{adjusted_gluten_out_dir}/kmer16_bray_curtis_complete.qza"

adjusted_gluten_metadata_path = f"{adjusted_gluten_out_dir}/metadata_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_permanova_dir = f"{vis_dir}/unadjusted_permanova"
adjusted_permanova_dir = f"{vis_dir}/adjusted_permanova"

asv_unadjusted_overall_path = f"{unadjusted_permanova_dir}/asv_overall.qzv"
kmer16_unadjusted_overall_path = f"{unadjusted_permanova_dir}/kmer16_overall.qzv"

asv_adjusted_overall_path = f"{adjusted_permanova_dir}/asv_overall.qzv"
kmer16_adjusted_overall_path = f"{adjusted_permanova_dir}/kmer16_overall.qzv"


# Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_gluten_out_dir"
!mkdir -p "$unadjusted_permanova_dir" "$adjusted_permanova_dir"

### 1.3 Load inputs

We load the whole-ASV and 16-mer Bray–Curtis distance matrices from notebook 07 and the cleaned metadata from notebook 06.

In [19]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
kmer_distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_distance_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

print("Whole-ASV samples:", len(asv_distance_matrix.ids))
print("16-mer samples:", len(kmer_distance_matrix.ids))
print("Metadata rows after exclusion:", len(metadata))

Whole-ASV samples: 150
16-mer samples: 150
Metadata rows after exclusion: 150


## 2. Select samples

### 2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [20]:
shared_ids = pd.Index(kmer_distance_matrix.ids).intersection(asv_distance_matrix.ids)
shared_ids = shared_ids.intersection(metadata.index)

matched = metadata.loc[shared_ids, columns].copy()

print("Matched samples:", len(matched))

Matched samples: 150


### 2.2 Keep complete cases

We keep samples labeled with sex, numeric age, settlement type, and recorded diet type. Both PERMANOVA models use this same subset.

In [21]:
matched["gluten_sample"] = matched["gluten_sample"].str.strip().replace("", pd.NA)
matched["age_years_sample"] = pd.to_numeric(matched["age_years_sample"], errors="coerce")
matched["sex_sample"] = matched["sex_sample"].str.strip().replace("", pd.NA)

gluten_complete = matched.dropna(subset=columns).copy()

gluten_groups = [
    "No",
    "I do not eat gluten because it makes me feel bad",
]

gluten_complete = gluten_complete.loc[
    gluten_complete["gluten_sample"].isin(gluten_groups)
].copy()

complete_ids = gluten_complete.index.tolist()

print("Complete-case samples:", len(complete_ids))

Complete-case samples: 136


We subset both Bray–Curtis matrices to the complete-case sample IDs in the same order and save them with the matching metadata.

In [22]:
asv_distance_complete = asv_distance_matrix.filter(complete_ids)
kmer16_distance_complete = kmer_distance_matrix.filter(complete_ids)

assert list(asv_distance_complete.ids) == complete_ids
assert list(kmer16_distance_complete.ids) == complete_ids

gluten_complete.index.name = "sample-id"
q2.Metadata(gluten_complete).save(adjusted_gluten_metadata_path)

q2.Artifact.import_data("DistanceMatrix", asv_distance_complete).save(
    adjusted_asv_gluten_distance_path
)
q2.Artifact.import_data("DistanceMatrix", kmer16_distance_complete).save(
    adjusted_kmer_gluten_distance_path
)
q2.Metadata(gluten_complete).save(adjusted_gluten_metadata_path)

print("Saved both distance matrices and metadata for", len(complete_ids), "samples.")

Saved both distance matrices and metadata for 136 samples.


### 2.3 Summarize samples

We compare sex group sizes before and after requiring complete metadata. We also check how the diet groups are distributed across sex types.

In [23]:
gluten_samples = matched.loc[matched["gluten_sample"].isin(gluten_groups)]

counts = (
    gluten_samples["gluten_sample"]
    .value_counts()
    .reindex(gluten_groups, fill_value=0)
    .to_frame("With gluten label")
)

counts["With all three variables"] = (
    gluten_complete["gluten_sample"]
    .value_counts()
    .reindex(gluten_groups, fill_value=0)
)

display(counts)

,With gluten label,With all three variables
gluten_sample,,
No,113,110
I do not eat gluten because it makes me feel bad,26,26


### 3.1 Unadjusted PERMANOVA

We test whether fungal community composition differs across BMI values. Both tests use the same 136 samples and include gluten consumption as the only predictor.

#### 3.1.1 Whole ASVs

In [24]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_asv_gluten_distance_path" \
    --m-metadata-file "$adjusted_gluten_metadata_path" \
    --p-formula "gluten_sample" \
    --o-visualization "$asv_unadjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/unadjusted_permanova/asv_overall.qzv


In [25]:
Visualization.load(asv_unadjusted_overall_path)

<visualization: Visualization uuid: 7f9b6118-efe7-4dc2-9340-3edfee0a485c>

#### 3.1.2 16-mers

In [26]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_kmer_gluten_distance_path" \
    --m-metadata-file "$adjusted_gluten_metadata_path" \
    --p-formula "gluten_sample" \
    --o-visualization "$kmer16_unadjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/unadjusted_permanova/kmer16_overall.qzv


In [27]:
Visualization.load(kmer16_unadjusted_overall_path)

<visualization: Visualization uuid: 42473a9e-43fe-40dd-9fa9-6c2998eb542f>

#### Unadjusted results

| **Gluten term** | **Whole ASVs** | **16-mers** |
| --------------- | -------------: | ----------: |
| Samples         |            136 |         136 |
| R²              |         0.0067 |      0.0063 |
| Pseudo-F        |          0.910 |       0.852 |
| p-value         |          0.501 |       0.537 |

Among the 136 samples included in both analyses, gluten consumption explained less than 1% of the variation in fungal community composition (0.67% for whole ASVs and 0.63% for 16-mers). Both p-values were above 0.05.

### 3.2 Adjusted PERMANOVA

We put settlement type last in the formula to test whether fungal community composition differs between the gluten-consuming and gluten-avoiding sample groups after accounting for age and sex. Both tests use the same 136 samples as the unadjusted tests.

#### 3.2.1 Whole ASVs

In [28]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_asv_gluten_distance_path" \
    --m-metadata-file "$adjusted_gluten_metadata_path" \
    --p-formula "age_years_sample + sex_sample + gluten_sample" \
    --o-visualization "$asv_adjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/adjusted_permanova/asv_overall.qzv


In [29]:
Visualization.load(asv_adjusted_overall_path)

<visualization: Visualization uuid: 476aa713-91bd-4453-b8b3-900e268107c6>

#### 3.2.2 16-mers

In [34]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_kmer_gluten_distance_path" \
    --m-metadata-file "$adjusted_gluten_metadata_path" \
    --p-formula "age_years_sample + sex_sample + gluten_sample" \
    --o-visualization "$kmer16_adjusted_overall_path"

Saved Visualization to: ../data/processed/13_beta_diversity_gluten/adjusted_permanova/kmer16_overall.qzv


In [35]:
Visualization.load(kmer16_adjusted_overall_path)

<visualization: Visualization uuid: fdbd3862-f973-46f9-9039-ea8dc79d07a1>

#### Adjusted results

| **Gluten term** | **Whole ASVs** | **16-mers** |
| --------------- | -------------: | ----------: |
| Samples         |            136 |         136 |
| R²              |         0.0053 |      0.0041 |
| Pseudo-F        |          0.719 |       0.561 |
| p-value         |          0.761 |       0.836 |

After accounting for age and sex, gluten consumption explained less than 1% of the variation in fungal community composition in both analyses (0.53% for whole ASVs and 0.41% for 16-mers). Both p-values were above 0.05.

### 4. Compare unadjusted and adjusted overall PERMANOVA

| **Bray–Curtis representation** | **Model**  | **Gluten R²** | **Pseudo-F** | **p-value** |
| ------------------------------ | ---------- | ------------: | -----------: | ----------: |
| Whole ASVs                     | Unadjusted |        0.0067 |        0.910 |       0.501 |
| Whole ASVs                     | Adjusted   |        0.0053 |        0.719 |       0.761 |
| 16-mers                        | Unadjusted |        0.0063 |        0.852 |       0.537 |
| 16-mers                        | Adjusted   |        0.0041 |        0.561 |       0.836 |

After adjustment, gluten R² decreased for both whole ASVs and 16-mers. Gluten consumption explained less than 1% of the variation in fungal community composition in all four tests, and all four p-values were above 0.05.